### This is where we chain every step

In [1]:
import sys
from pathlib import Path
import yaml
import pandas as pd
project_dir = Path.cwd().parent
sys.path.append(str(project_dir))
config_path = project_dir / "config.yaml"

from src.corrupt import random_missing_values
from src.clean import mean_fill
from src.evaluate import evaluate_continuous_recovery
from src.load import load_model_data
from src.run_experiment import run_experiment

with config_path.open() as config_file:
    config = yaml.safe_load(config_file)

corruption_rate = config["corruption_rate"]
target_drug = config["target_drug"]
columns = config["columns"]
seed = config["seed"]

processed_dir = project_dir / "data/processed/"
drug_consumption = project_dir / f"data/processed/Drug_Consumption_{target_drug}.csv"

# load the dfs
original_df = load_model_data(drug_consumption)

# corrupt the df
corrupted_df, mask_df = random_missing_values(original_df, corruption_rate, seed)

# clean the df 
cleaned_df = mean_fill(corrupted_df, columns, {})

# evaluate the cleaned df
evaluated_metrics = evaluate_continuous_recovery(original_df, cleaned_df, mask_df, columns)

# print("Continuous Recovery Metrics")
# print("---------------------------")
# print(f"RMSE:             {evaluated_metrics['rmse']:.4f}")
# print(f"MAE:              {evaluated_metrics['mae']:.4f}")
# print(f"Bias:             {evaluated_metrics['bias']:.4f}")
# print(f"Variance Ratio:   {evaluated_metrics['variance_ratio']:.4f}")
# print(f"Wasserstein:      {evaluated_metrics['wasserstein']:.4f}")

result_df = run_experiment(original_df)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

parquet = pd.read_parquet(project_dir / "results/experiments.parquet")

parquet

,seed,corruption_rate,strategy,corruption_type,recovery_applicable,rmse,mae,bias,variance_ratio,wasserstein
0,1,0.1,mean_fill,random_missing_values,True,0.994414,0.799872,0.046914,0.000053,0.793510
1,1,0.1,do_nothing,random_missing_values,False,NaN,NaN,NaN,NaN,NaN
2,1,0.1,knn_impute,random_missing_values,True,1.021971,0.811291,0.025534,0.400479,0.302145
3,1,0.1,mean_fill,mnar_missing_values,True,1.006205,0.797980,0.007642,0.000094,0.790182
4,1,0.1,do_nothing,mnar_missing_values,False,NaN,NaN,NaN,NaN,NaN
5,1,0.1,knn_impute,mnar_missing_values,True,0.981520,0.778166,-0.003049,0.379505,0.301741
6,1,0.2,mean_fill,random_missing_values,True,0.987827,0.795284,0.017840,0.000083,0.787552
7,1,0.2,do_nothing,random_missing_values,False,NaN,NaN,NaN,NaN,NaN
8,1,0.2,knn_impute,random_missing_values,True,1.059677,0.845621,0.035541,0.351748,0.330551
9,1,0.2,mean_fill,mnar_missing_values,True,0.981888,0.781077,0.019973,0.000160,0.771735
